<a href="https://colab.research.google.com/github/Rumas0/Thesis_work_SSL-imbalance/blob/main/SSL_Only_No_Rebalance_Ablation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import pandas as pd
import numpy as np
import os
import zipfile
import shutil
from sklearn.metrics import accuracy_score, classification_report
from collections import Counter
import json

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

Mounted at /content/drive
Device: cuda


#**Setup & Extraction**

In [4]:
BACKUP_DIR = '/content/drive/MyDrive/Thesis-work/Backups/thesis_backup_day1'
ZIP_PATH = '/content/drive/MyDrive/Thesis-work/ISIC_2019_Training_Input.zip'
UNLABELED_DIR = 'data/isic2019/ISIC_2019_Training_Input'
LABELED_DIR = 'data/labeled_real'

os.makedirs('data/isic2019', exist_ok=True)
os.makedirs(LABELED_DIR, exist_ok=True)

if not os.path.exists(UNLABELED_DIR) or len(os.listdir(UNLABELED_DIR)) <1000:
  with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
    zip_ref.extractall('data/isic2019/')

##Filter Labeled Images
train_df = pd.read_csv(f'{BACKUP_DIR}/expA_train.csv')
val_df = pd.read_csv(f'{BACKUP_DIR}/expA_val.csv')
test_df = pd.read_csv(f'{BACKUP_DIR}/expA_test.csv')
all_images = pd.concat([train_df, val_df, test_df])['image'].unique()

for img_id in all_images:
  src = f'{UNLABELED_DIR}/{img_id}.jpg'
  dst = f'{LABELED_DIR}/{img_id}.jpg'
  if os.path.exists(src):
    shutil.copy(src, dst)

**Load SSL Encoder (pr-etrained on 24k real images)**

In [6]:
class SimpleEncoder(nn.Module):
  def __init__(self):
    super().__init__()
    self.features = nn.Sequential(
        nn.Conv2d(3, 32, 3,padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
        nn.Conv2d(32, 64, 3,padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
        nn.Conv2d(64, 128, 3,padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2),
        nn.AdaptiveAvgPool2d(1), nn.Flatten()
    )

  def forward(self, x):
    return self.features(x)

class SSLClassifier(nn.Module):
  def __init__(self, encoder, num_classes):
    super().__init__()
    self.encoder = encoder
    self.classifier = nn.Sequential(nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.3),
    nn.Linear(64, num_classes)
    )

  def forward(self, x):
    return self.classifier(self.encoder(x))

##Loading the SSL encoder just trained
encoder = SimpleEncoder()
ssl_path = f'{BACKUP_DIR}/ssl_encoder_real_24000.pth'
encoder.load_state_dict(torch.load(ssl_path, map_location=device))
print(f'Loaded SSL Encoder from:{ssl_path}')

##Unfreeze All for Fine-tuning
for param in encoder.parameters():
  param.requires_grad = True

model = SSLClassifier(encoder, 3).to(device) ## 3 Classes: BKL, MEL, NV

Loaded SSL Encoder from:/content/drive/MyDrive/Thesis-work/Backups/thesis_backup_day1/ssl_encoder_real_24000.pth


**STANDARD DATASET (NO augmentation, NO oversampling)**

In [7]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

class StandardDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df
        self.transform = transform
        self.classes = sorted(df['label'].unique())
        self.class_to_idx = {c:i for i,c in enumerate(self.classes)}
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(f"{LABELED_DIR}/{row['image']}.jpg").convert('RGB')
        label = self.class_to_idx[row['label']]
        if self.transform:
            img = self.transform(img)
        return img, label

train_ds = StandardDataset(train_df, transform)
val_ds = StandardDataset(val_df, test_transform)
test_ds = StandardDataset(test_df, test_transform)

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=16)
test_loader = DataLoader(test_ds, batch_size=16)

print(f"Train: {len(train_ds)} (original, no oversampling)")
print(f"Class distribution: {train_df['label'].value_counts().to_dict()}")

Train: 483 (original, no oversampling)
Class distribution: {'NV': 349, 'BKL': 99, 'MEL': 35}


**PLAIN CROSS-ENTROPY (NO class weights, NO Focal Loss)**

In [8]:
criterion = nn.CrossEntropyLoss() ##Plain CE - no rebalancing
optimizer = optim.Adam(model.parameters(), lr=0.0001)

print("\n Using Plain CrossEntropyLoss (No rebalancing)")



 Using Plain CrossEntropyLoss (No rebalancing)


#**Training & Evaluation**

In [9]:
def train_epoch(model, loader):
    model.train()
    total_loss, correct = 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()
        correct += (outputs.argmax(1) == labels).sum().item()
    return total_loss / len(loader), correct / len(loader.dataset)

def evaluate(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            outputs = model(images)
            preds = outputs.argmax(1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())
    return accuracy_score(all_labels, all_preds), all_labels, all_preds

print(f"\n{'='*36}")
print("ABLATION 1: SSL ONLY (NO rebalancing)")
print(f"{'='*36}")

epochs = 30
best_val = 0
patience = 7
epochs_no_improve = 0

for epoch in range(epochs):
    train_loss, train_acc = train_epoch(model, train_loader)
    val_acc, _, _ = evaluate(model, val_loader)

    if val_acc > best_val:
        best_val = val_acc
        torch.save(model.state_dict(), 'best_ssl_only.pth')
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1

    if (epoch + 1) % 3 == 0:
        print(f"Epoch {epoch+1}: Train={train_acc:.3f}, Val={val_acc:.3f}")

    if epochs_no_improve >= patience:
        print(f"Early stopping at epoch {epoch+1}")
        break

## Evaluate
model.load_state_dict(torch.load('best_ssl_only.pth', map_location=device))
test_acc, true_labels, pred_labels = evaluate(model, test_loader)

print("\n" + "="*36)
print("Results: SSL Only (NO rebalancing)")
print("="*36)
print(f"Test Accuracy: {test_acc:.3f}")

report = classification_report(true_labels, pred_labels,
                                target_names=train_ds.classes, output_dict=True)
print(classification_report(true_labels, pred_labels, target_names=train_ds.classes))

mel_recall = report['MEL']['recall']
print(f"\n===>>> MEL Recall: {mel_recall:.1%} <<<===")
print(f"Predictions: {Counter(pred_labels)}")

## Save
results = {
    'experiment': 'SSL_Only_No_Rebalance',
    'ssl': True,
    'rebalancing': False,
    'augmentation': False,
    'test_accuracy': test_acc,
    'mel_recall': mel_recall,
    'report': report
}
with open('ssl_only_results.json', 'w') as f:
    json.dump(results, f, indent=2)

torch.save(model.state_dict(), f'{BACKUP_DIR}/ssl_only_finetuned.pth')
print("\n Saved to Drive")


ABLATION 1: SSL ONLY (NO rebalancing)
Epoch 3: Train=0.727, Val=0.721
Epoch 6: Train=0.731, Val=0.721
Epoch 9: Train=0.737, Val=0.731
Early stopping at epoch 9

Results: SSL Only (NO rebalancing)
Test Accuracy: 0.721
              precision    recall  f1-score   support

         BKL       0.50      0.05      0.09        21
         MEL       0.00      0.00      0.00         8
          NV       0.73      0.99      0.84        75

    accuracy                           0.72       104
   macro avg       0.41      0.34      0.31       104
weighted avg       0.62      0.72      0.62       104


===>>> MEL Recall: 0.0% <<<===
Predictions: Counter({np.int64(2): 102, np.int64(0): 2})

 Saved to Drive


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/m